# Capítulo 30 · Laboratório 30: psicologia do trader em números
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026** (40 ações desde 2005, seis ações que caíram mais de 90%, Ibovespa desde 1993 e S&P 500 desde 1950). Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap30.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. As 40 ações passam pela limpeza do Capítulo 28; o método das Partes 4 a 7 é o mesmo do Capítulo 29 (rompimento de 20 dias com stop de 3 ATR). Custo de 0,10% por operação. Os sorteios usam semente fixa: os números se repetem a cada execução. A Parte 8 cria o arquivo `diario_modelo.csv` para você preencher.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import re
import time
import numpy as np
import pandas as pd
import yfinance as yf

B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
# ações que caíram mais de 90% do topo e não voltaram (ficam fora da amostra principal: viés de sobrevivência)
CAIDAS = ["AMER3.SA", "MGLU3.SA", "BHIA3.SA", "IRBR3.SA", "CVCB3.SA", "OIBR3.SA"]
FIM, CUSTO = "2026-10-01", 0.001
rng = np.random.default_rng(32)            # todos os sorteios (semente fixa: os números se repetem)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
CACHE = "dados_cap30.pkl"


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


def baixa(*args, **kw):
    for tentativa in range(5):                 # o provedor às vezes falha: tenta de novo
        try:
            x = yf.download(*args, progress=False, **kw)
            if len(x):
                return x
        except Exception:
            pass
        time.sleep(5 * (tentativa + 1))
    ativo = args[0] if isinstance(args[0], str) else f"a lista de {len(args[0])} ativos"
    raise RuntimeError(f"Não consegui baixar {ativo} do Yahoo Finance depois de 5 tentativas. O provedor pode estar"
                       " fora do ar ou limitando pedidos. Espere alguns minutos e rode a célula de novo.")


if os.path.exists(CACHE):                  # cópia local: o provedor pode corrigir o histórico entre execuções
    dados, ibov, spx, caidas = pd.read_pickle(CACHE)
else:
    dados = baixa(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, group_by="ticker")
    ibov = baixa("^BVSP", start="1993-01-01", end=FIM, auto_adjust=True, multi_level_index=False)
    spx = baixa("^GSPC", start="1950-01-01", end=FIM, auto_adjust=True, multi_level_index=False)
    caidas = baixa(CAIDAS, start="2005-01-01", end=FIM, auto_adjust=True, group_by="ticker")
    pd.to_pickle((dados, ibov, spx, caidas), CACHE)


CORRIGIDAS = {}                            # o que a limpeza mudou (registro da Parte 8; sem repetir)
SUSPEITOS = {"EQTL3.SA": ("2010-08-25", "queda de 40% sem volta (provável ajuste de bonificação ou desdobramento faltando)")}


def limpa(d, nome=""):
    """Limpeza do Capítulo 26, mais três regras para dado sujo que engana o teste de stops:
    1) máxima impossível: pavio de mais de 50% acima da abertura e do fechamento vira o maior dos dois;
    2) desdobramento ou grupamento sem ajuste: salto de mais de 2 vezes (ou menos da metade) da abertura em relação ao
       fechamento anterior, sem volta nos 5 pregões seguintes, corrige o histórico anterior pela mesma proporção;
    3) buraco nos dados: com mais de 15 dias corridos sem cotação, só fica o trecho contínuo depois do último buraco."""
    d = d.dropna().copy()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"])
    d = d[ok & (d["Volume"] > 0)] if "Volume" in d else d[ok]
    px = ["Open", "High", "Low", "Close"]
    topo = d[["Open", "Close"]].max(axis=1)
    alta = d["High"] > 1.5 * topo
    for dt in d.index[alta]:
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: máxima {d.at[dt, 'High']:.2f} com fechamento {d.at[dt, 'Close']:.2f}", None)
    d.loc[alta, "High"] = topo[alta]
    salto = d["Open"] / d["Close"].shift()
    dura = d["Close"].shift(-5) / d["Close"].shift()
    for dt in d.index[((salto < 0.5) | (salto > 2)) & ((dura < 0.6) | (dura > 1.6))]:
        f_ = salto.loc[dt]
        d.loc[d.index < dt, px] *= f_
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: salto de {f_:.4f} vezes sem volta (ajuste que faltou); histórico"
                          " anterior corrigido", None)
    if nome == "SUZB3.SA":                 # antes disso, preço parado no Yahoo (Capítulo 26)
        d = d.loc["2017-11-10":]
    if nome in SUSPEITOS:                  # queda sem volta que parece ajuste faltando, mas sem prova: fica fora
        dt, motivo = SUSPEITOS[nome]
        CORRIGIDAS.setdefault(f"{nome} {pd.Timestamp(dt):%d/%m/%Y}: {motivo}; usa só a partir dessa data", None)
        d = d.loc[dt:]
    buraco = d.index.to_series().diff().dt.days
    if (buraco > 15).any():
        ini = buraco[buraco > 15].index[-1]
        CORRIGIDAS.setdefault(f"{nome}: {int((buraco > 15).sum())} buracos de mais de 15 dias; usa só a partir de {ini:%d/%m/%Y}", None)
        d = d.loc[ini:]
    return d


def limpa_simples(d):
    """Para as ações que caíram mais de 90%: só tira barras vazias. Os saltos ali são quedas reais (a Americanas abriu
    77% abaixo em 12/01/2023), e a regra do salto sem ajuste as apagaria."""
    d = d.dropna()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"]) & (d["Volume"] > 0)
    return d[ok]


def atr(h, l, c, n=20):
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()


BASE = {t: limpa(dados[t], t) for t in TICKERS}

## Parte 1: aversão à perda

In [ ]:
ALFA, LAMBDA = 0.88, 2.25          # parâmetros medianos de Tversky e Kahneman (1992)


def valor(x):
    x = np.asarray(x, dtype=float)
    return np.where(x >= 0, np.abs(x) ** ALFA, -LAMBDA * np.abs(x) ** ALFA)


g_min = 1000 * LAMBDA ** (1 / ALFA)
mostra("Parte 1. Aversão à perda (função valor de Tversky e Kahneman, 1992: alfa 0,88, lambda 2,25)")
mostra(f"   ganhar R$ 1_000 'vale' {valor(1000):.0f}; perder R$ 1_000 'dói' {valor(-1000):.0f} (a dor pesa {LAMBDA:.2f} vezes)")
mostra(f"   cara ou coroa: para topar arriscar R$ 1_000, a pessoa típica pede um prêmio de R$ {g_min:_.0f}")
mostra(f"   sem a aversão (lambda = 1) bastaria R$ 1_000")

## Parte 2: efeito disposição

In [ ]:
ALVO, STOP, MAXD = 0.05, 0.05, 250


def disposicao(base, n, filtra=True):
    """Mesmas entradas sorteadas, duas condutas. filtra=True descarta janelas com barra suspeita (variação > 40% num dia)."""
    res, tick = [], rng.choice(list(base), n)
    for t, d in base.items():
        O, H, L, C = (d[x].to_numpy() for x in ("Open", "High", "Low", "Close"))
        prev = np.r_[np.nan, C[:-1]]
        susp = (np.abs(C / prev - 1) > 0.4) | (np.abs(O / prev - 1) > 0.4)
        m20 = d["Low"].rolling(20).min().shift(1).to_numpy()
        for i in rng.integers(30, len(C) - MAXD - 2, int((tick == t).sum())):
            if filtra and susp[i:i + MAXD + 1].any():
                continue
            e = C[i]
            # (A) disposição: vende no alvo de +5%; a perdedora fica até voltar ao preço pago (ou 250 pregões)
            sa, da, tipo = None, None, "presa"
            for k in range(i + 1, i + MAXD + 1):
                if H[k] >= e * (1 + ALVO):
                    sa, da, tipo = max(e * (1 + ALVO), O[k]), k - i, "alvo"; break
                if C[k - 1] < e <= H[k]:
                    sa, da, tipo = max(e, O[k]), k - i, "empate"; break
            if sa is None:
                sa, da = C[i + MAXD], MAXD
            # (B) o contrário: stop de -5%; a ganhadora segue até perder a mínima de 20 pregões (ou 250 pregões)
            sb, db, st = None, None, e * (1 - STOP)
            for k in range(i + 1, i + MAXD + 1):
                if L[k] <= st:
                    sb, db = min(st, O[k]), k - i; break
                st = max(st, m20[k]) if C[k] > e * (1 + STOP) else st
            if sb is None:
                sb, db = C[i + MAXD], MAXD
            res.append((t, sa / e - 1 - CUSTO, da, tipo, sb / e - 1 - CUSTO, db))
    return pd.DataFrame(res, columns=["ativo", "rA", "dA", "tipoA", "rB", "dB"])


def linha(r, dias):
    return (f"média {r.mean():+.2%} | mediana {np.median(r):+.2%} | pior 1% {np.percentile(r, 1):+.0%} | pior {r.min():+.0%}"
            f" | pregões médios {dias.mean():.0f}")


RES_DISP = {}
for nome, base, n in [("40 ações da amostra (sobreviventes)", BASE, 6000),
                      ("6 ações que caíram mais de 90% (AMER3, MGLU3, BHIA3, IRBR3, CVCB3, OIBR3)",
                       {t: limpa_simples(caidas[t]) for t in CAIDAS}, 6000)]:
    D = disposicao(base, n, filtra=(base is BASE))
    RES_DISP[nome[:2].strip()] = D
    q = D.tipoA.value_counts(normalize=True)
    pr = D[D.tipoA == "presa"]
    mostra(f"\nParte 2. Efeito disposição. {nome}: {len(D):_} entradas sorteadas, custo 0,10%")
    mostra(f"   A) vende o ganhador em +5% e segura o perdedor até voltar ao preço pago: {linha(D.rA.to_numpy(), D.dA.to_numpy())}")
    mostra(f"      saiu no alvo {q.get('alvo', 0):.0%} | 'saiu no zero a zero' {q.get('empate', 0):.0%} | ficou preso um ano {q.get('presa', 0):.0%}"
          f" (resultado médio dos presos {pr.rA.mean():+.0%})")
    mostra(f"   B) stop de -5% e deixa o ganhador correr: {linha(D.rB.to_numpy(), D.dB.to_numpy())}")
    mostra(f"      acerto {np.mean(D.rB > 0):.0%} | ganho médio {D.rB[D.rB > 0].mean():+.1%} | perda média {D.rB[D.rB <= 0].mean():+.1%}")

## Parte 3: falácia do apostador

In [ ]:
mostra("\nParte 3. Depois de k quedas seguidas, qual a chance de o dia seguinte ser de alta?")
for nome, s in [("Ibovespa, de julho de 1994 (Plano Real) a 2026", ibov["Close"].loc["1994-07-01":]), ("S&P 500, de 1950 a 2026", spx["Close"])]:
    r = s.pct_change().dropna()
    up = (r > 0).astype(int).to_numpy()
    seq = np.zeros(len(up), int)                      # quedas seguidas até o dia t
    for j in range(len(up)):
        seq[j] = 0 if up[j] else (seq[j - 1] + 1 if j else 1)
    txt = [f"todos os dias {up.mean():.1%}"]
    for kq in range(1, 7):
        idx = np.flatnonzero(seq[:-1] == kq) if kq < 6 else np.flatnonzero(seq[:-1] >= kq)
        txt.append(f"{kq}{'+' if kq == 6 else ''}: {up[idx + 1].mean():.1%} (n = {len(idx):_})")
    mostra(f"   {nome}: " + " | ".join(txt))

## Parte 4: sequências de perdas

In [ ]:
ops = []
for t in TICKERS:
    d = BASE[t]
    O, H, L, C = (d[x].to_numpy() for x in ("Open", "High", "Low", "Close"))
    A = atr(d["High"], d["Low"], d["Close"]).to_numpy()
    c = d["Close"]
    sinal = ((c > d["High"].rolling(20).max().shift(1)) & (c > c.rolling(200).mean())).to_numpy()
    livre = 0
    for i in np.flatnonzero(sinal):
        if i < 210 or i > len(C) - 22 or i < livre:
            continue
        e, risco = C[i], 3 * A[i]
        saida, fim = None, i + 20
        for k in range(i + 1, i + 21):
            if L[k] <= e - risco:
                saida, fim = min(e - risco, O[k]), k; break
        saida = C[i + 20] if saida is None else saida
        mfe = (H[i + 1:fim + 1].max() - e) / risco          # maior lucro aberto durante a operação, em R
        ops.append((d.index[i], t, (saida - e) / risco - CUSTO * e / risco, mfe))
        livre = i + 20
P = pd.DataFrame(ops, columns=["data", "ativo", "R", "MFE"]).sort_values("data", kind="stable").reset_index(drop=True)
R = P["R"].to_numpy()


def maior_seq(perdeu):
    m = c_ = 0
    for p in perdeu:
        c_ = c_ + 1 if p else 0
        m = max(m, c_)
    return m


mostra(f"\nParte 4. Sequências de perdas. Método do Capítulo 29 (rompimento de 20 dias, stop de 3 ATR): {len(R):_} operações,"
      f" acerto {np.mean(R > 0):.0%}, expectativa {R.mean():+.3f} R")
mostra(f"   maior sequência de perdas na ordem real das datas: {maior_seq(R <= 0)}")
for acerto in (0.40, 0.50, 0.60):
    for n in (100, 250):
        m = [maior_seq(rng.random(n) > acerto) for _ in range(5000)]
        mostra(f"   acerto {acerto:.0%}, {n} operações: maior sequência de perdas mediana {np.median(m):.0f},"
              f" em 1 de cada 10 contas chega a {np.percentile(m, 90):.0f} ou mais")

## Parte 5: abandonar o método e dobrar a mão

In [ ]:
RISCO = 0.01


def segue(r, risco=RISCO):
    return np.cumprod(1 + risco * r)


def abandona(r, risco=RISCO, queda=0.10):
    """Para de operar quando a conta cai 10% do topo; volta só quando o método, no papel, faz novo topo."""
    papel = segue(r, risco)
    conta, topo_conta, fora, marca = [1.0], 1.0, False, 0.0
    for j, x in enumerate(r):
        if fora:
            conta.append(conta[-1])
            if papel[j] > marca:
                fora, topo_conta = False, conta[-1]
            continue
        conta.append(conta[-1] * (1 + risco * x))
        topo_conta = max(topo_conta, conta[-1])
        if conta[-1] / topo_conta - 1 <= -queda:
            fora, marca = True, papel[:j + 1].max()
    return np.array(conta[1:])


def martingale(r, risco=RISCO):
    """Dobra o risco depois de cada perda e volta a 1% depois do ganho (risco limitado a 100% da conta)."""
    conta, f = [1.0], risco
    for x in r:
        conta.append(max(conta[-1] * (1 + min(f, 1.0) * x), 0.0))
        f = risco if x > 0 else f * 2
    return np.array(conta[1:])


def pior_queda(eq):
    eq = np.r_[1.0, eq]
    return (eq / np.maximum.accumulate(eq) - 1).min()


N_OP, N_SIM = 250, 5000
A_, B_, C_, fora_, qA, qC = [], [], [], [], [], []
for _ in range(N_SIM):
    rr = rng.choice(R, N_OP)
    ea, eb, ec = segue(rr), abandona(rr), martingale(rr)
    A_.append(ea[-1]); B_.append(eb[-1]); C_.append(ec[-1]); qA.append(pior_queda(ea)); qC.append(pior_queda(ec))
    fora_.append(np.mean(np.r_[1.0, eb[:-1]] == eb))
A_, B_, C_, fora_, qA, qC = map(np.array, (A_, B_, C_, fora_, qA, qC))
parou = fora_ > 0
p6 = np.mean([maior_seq(rng.choice(R, N_OP) <= 0) >= 6 for _ in range(N_SIM)])
mostra(f"\nParte 5. {N_SIM:_} sequências de {N_OP} operações sorteadas do método, risco de 1% por operação")
mostra(f"   segue o método: mediana {np.median(A_):.2f}x | termina no prejuízo {np.mean(A_ < 1):.0%} | cai 50% ou mais em algum momento {np.mean(qA <= -0.5):.0%}")
mostra(f"   abandona na queda de 10% e só volta no novo topo do método: parou em {parou.mean():.0%} das sequências;"
      f" entre essas, termina pior que quem seguiu em {np.mean(B_[parou] < A_[parou]):.0%}"
      f" (mediana {np.median(B_[parou] / A_[parou]) - 1:+.0%} em relação a quem seguiu)")
mostra(f"   dobra a mão depois de cada perda (Martingale): mediana {np.median(C_):.2f}x | cai 50% ou mais {np.mean(qC <= -0.5):.0%}"
      f" | cai 80% ou mais {np.mean(qC <= -0.8):.0%} | termina com menos de 10% da conta {np.mean(C_ < 0.1):.0%}")
mostra(f"   depois de 6 perdas seguidas o Martingale arrisca 64% da conta; chance de 6 ou mais perdas seguidas em {N_OP} operações: {p6:.0%}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.hist(np.log10(A_), bins=60, alpha=0.6, label="segue o método")
plt.hist(np.log10(np.clip(C_, 1e-3, None)), bins=60, alpha=0.6, label="Martingale")
plt.xlabel("capital final (log10; 0 = empatou)")
plt.ylabel("sequências")
plt.legend()
plt.show()

## Parte 6: auditoria das 30 operações

In [ ]:
blocos = np.array([R[j:j + 30].mean() for j in range(0, len(R) - 29, 30)])
boot = rng.choice(R, (20000, 30)).mean(axis=1)
mostra(f"\nParte 6. Blocos de 30 operações seguidas: {len(blocos)} blocos; média por bloco de {blocos.min():+.2f} R a {blocos.max():+.2f} R;"
      f" {np.mean(blocos <= 0):.0%} dos blocos no prejuízo")
mostra(f"   sorteando 30 operações do método: {np.mean(boot <= 0):.0%} de chance de um bloco negativo,"
      f" mesmo com expectativa de {R.mean():+.3f} R")

## Parte 7: devolver o lucro aberto

In [ ]:
mostra("\nParte 7. Devolver o lucro: operações que chegaram a ter lucro aberto e terminaram no zero ou no prejuízo")
for nivel in (1.0, 2.0):
    m = P["MFE"] >= nivel
    mostra(f"   chegaram a +{nivel:.0f} R em algum momento: {m.mean():.0%} das operações;"
          f" dessas, {np.mean(P.loc[m, 'R'] <= 0):.0%} terminaram no zero ou no prejuízo"
          f" e {np.mean(P.loc[m, 'R'] < P.loc[m, 'MFE'] / 2):.0%} devolveram mais da metade do pico")
mostra(f"   lucro aberto médio devolvido (pico menos resultado): {(P['MFE'].clip(lower=0) - P['R']).mean():.2f} R por operação")

## Parte 8: diário de operações

In [ ]:
COLUNAS = ["data", "ativo", "setup", "entrada", "stop", "saida", "R", "seguiu_plano", "emocao", "nota"]
diario = P.assign(setup="rompimento 20d", entrada=np.nan, stop=np.nan, saida=np.nan,
                  seguiu_plano=pd.NA, emocao=pd.NA, nota="")[COLUNAS]


def relatorio(df, por):
    g = df.dropna(subset=[por]).groupby(por)["R"]
    return pd.DataFrame({"operacoes": g.size(), "acerto": g.apply(lambda x: (x > 0).mean()),
                         "expectativa_R": g.mean(), "soma_R": g.sum()}).round(3)


diario["mercado"] = np.where(diario.ativo.str.endswith(".SA"), "B3", "EUA")
diario["ano"] = diario.data.dt.year
mostra("\nParte 8. Diário: o mesmo relatório serve para 'seguiu_plano' e 'emocao' quando o leitor preencher")
mostra(relatorio(diario, "mercado").to_string())
mostra("   preenchidas até agora: seguiu_plano", int(diario.seguiu_plano.notna().sum()), "| emocao", int(diario.emocao.notna().sum()))
diario[COLUNAS].head(0).to_csv("diario_modelo.csv", index=False)
mostra(f"   o que a limpeza corrigiu nas 40 ações ({len(CORRIGIDAS)} casos):")
for linha_ in CORRIGIDAS:
    mostra("      " + linha_)

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `maior_seq` da Parte 4 e a função `martingale` da Parte 5. Mostre por que `f = risco if x > 0 else f * 2` leva o risco de 1% a 64% da conta depois de seis perdas seguidas." Confira com a Parte 5, que imprime a mediana de 2,47 vezes do Martingale contra 1,41 vez de quem segue o método, e a queda de 80% ou mais em 31% das sequências.
2. **Adaptar:** "Leia o meu `diario_modelo.csv` preenchido, ordene por data e rode `maior_seq` na coluna R, contando como perda o R menor ou igual a zero. Depois rode `relatorio(diario, 'seguiu_plano')`. Não mude as duas funções." Antes de ler o seu diário, confira se a Parte 4 continua mostrando a maior sequência de 17 perdas na ordem real das datas e se a Parte 8 mostra 1.584 operações na B3 e 2.204 nos Estados Unidos, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu caso) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque o alvo de +5% por +10% na conduta A. Ela melhora nas ações que caíram?
2. Na Parte 3, troque o Ibovespa por uma ação de sua escolha. A alta "está devendo" depois de cinco quedas?
3. Na Parte 5, troque a queda de 10% que faz o trader abandonar o método por 20%. Ele ainda sai na hora errada?
4. Na Parte 8, preencha 30 operações suas no diario_modelo.csv e rode relatorio(diario, "seguiu_plano").

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.